# Chapter 11.4: Performance Profiling Workflow

This notebook covers systematic performance profiling for vLLM and SGLang,
from methodology to tools to analysis.

**Learning Objectives:**
- Understand profiling methodology for LLM serving
- Use torch.profiler for trace analysis
- Use Nsight Systems and Nsight Compute
- Build custom profiling hooks
- Identify and categorize performance bottlenecks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part11/chapter_11.4_profiling.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part11/chapter_11.4_profiling.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Profiling Methodology

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

def draw_profiling_tools_comparison():
    """Profiling tools comparison: py-spy -> torch.profiler -> Nsight Systems -> Nsight Compute."""
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.set_xlim(0, 18); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title('Profiling Tools Comparison: From High-Level to Low-Level',
                 fontsize=15, fontweight='bold', pad=15)

    tools = [
        {
            'name': 'py-spy', 'level': 'Python Level',
            'detail': 'Sampling profiler\nNo code changes\nFlame graphs',
            'color': '#4A90D9', 'scope': 'Which Python\nfunctions are slow?',
        },
        {
            'name': 'torch.profiler', 'level': 'PyTorch Ops',
            'detail': 'Op-level timing\nCPU + CUDA\nChrome trace export',
            'color': '#27AE60', 'scope': 'Which operators\n(mm, attn) dominate?',
        },
        {
            'name': 'Nsight Systems', 'level': 'System Level',
            'detail': 'GPU timeline\nKernel launches\nMemory transfers',
            'color': '#F39C12', 'scope': 'GPU idle gaps?\nCPU-GPU overlap?',
        },
        {
            'name': 'Nsight Compute', 'level': 'Kernel Level',
            'detail': 'Per-kernel analysis\nRoofline model\nWarp stall reasons',
            'color': '#E74C3C', 'scope': 'Is this kernel\ncompute or mem bound?',
        },
    ]

    # Draw as layered boxes from left (high-level) to right (low-level)
    for i, tool in enumerate(tools):
        x = 0.5 + i * 4.3
        # Main box
        rect = FancyBboxPatch((x, 3.0), 3.8, 4.5, boxstyle='round,pad=0.15',
                              facecolor=tool['color'], edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        # Name
        ax.text(x + 1.9, 6.8, tool['name'], ha='center', va='center',
                fontsize=13, fontweight='bold', color='white')
        # Level label
        ax.text(x + 1.9, 6.0, tool['level'], ha='center', va='center',
                fontsize=9, color='white', alpha=0.9)
        # Detail
        ax.text(x + 1.9, 4.8, tool['detail'], ha='center', va='center',
                fontsize=8, color='white', alpha=0.9)
        # Scope question
        ax.text(x + 1.9, 2.3, tool['scope'], ha='center', va='center',
                fontsize=8, color=tool['color'], fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=tool['color'], alpha=0.9))

        # Arrow to next
        if i < len(tools) - 1:
            ax.annotate('', xy=(x + 4.0, 5.25), xytext=(x + 4.3, 5.25),
                        arrowprops=dict(arrowstyle='<-', color='#333', lw=2))

    # Top label: granularity arrow
    ax.annotate('', xy=(17, 8.5), xytext=(1, 8.5),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))
    ax.text(9, 9.0, 'Increasing Granularity & Overhead', ha='center',
            fontsize=11, fontweight='bold', color='#333')
    ax.text(1, 8.0, 'Low overhead\nBroad view', fontsize=8, color='#4A90D9')
    ax.text(15, 8.0, 'High overhead\nDeep analysis', fontsize=8, color='#E74C3C', ha='right')

    plt.tight_layout()
    plt.savefig('profiling_tools_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_profiling_tools_comparison()

In [ ]:
# Demo: Profiling methodology - what to measure and when

methodology = """
Performance Profiling Methodology for LLM Serving
==================================================

Key Metrics to Measure:

1. Latency Metrics:
   - Time to First Token (TTFT): Time from request to first output token
   - Inter-Token Latency (ITL): Time between consecutive output tokens
   - End-to-End Latency: Total time from request to completion

2. Throughput Metrics:
   - Tokens per second (total output tokens / wall time)
   - Requests per second
   - GPU utilization percentage

3. Resource Metrics:
   - GPU memory usage (peak and average)
   - GPU compute utilization
   - Memory bandwidth utilization
   - CPU utilization
   - Network bandwidth (for distributed)

Profiling Phases:
  Phase 1: Macro profiling  - Overall system metrics (tokens/s, latency)
  Phase 2: Component timing - Per-component breakdown (attention, MLP, etc.)
  Phase 3: Kernel profiling - Individual CUDA kernel analysis
  Phase 4: Memory profiling - Memory allocation and bandwidth

Profiling Workflow:
  1. Establish baseline with default config
  2. Identify the dominant bottleneck
  3. Drill down into the bottleneck
  4. Apply optimization
  5. Measure improvement
  6. Repeat
"""

print(methodology)

In [ ]:
# Demo: Metrics collection framework
import time
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import statistics

@dataclass
class RequestMetrics:
    request_id: str
    prompt_tokens: int
    output_tokens: int
    start_time: float
    first_token_time: Optional[float] = None
    end_time: Optional[float] = None
    token_timestamps: List[float] = field(default_factory=list)
    
    @property
    def ttft(self):
        if self.first_token_time:
            return self.first_token_time - self.start_time
        return None
    
    @property
    def e2e_latency(self):
        if self.end_time:
            return self.end_time - self.start_time
        return None
    
    @property
    def inter_token_latencies(self):
        if len(self.token_timestamps) < 2:
            return []
        return [self.token_timestamps[i+1] - self.token_timestamps[i]
                for i in range(len(self.token_timestamps) - 1)]

class ProfilingCollector:
    """Collects and analyzes serving performance metrics."""
    
    def __init__(self):
        self.metrics: List[RequestMetrics] = []
    
    def add_metrics(self, m: RequestMetrics):
        self.metrics.append(m)
    
    def summary(self) -> Dict:
        if not self.metrics:
            return {}
        
        ttfts = [m.ttft for m in self.metrics if m.ttft is not None]
        e2e_lats = [m.e2e_latency for m in self.metrics if m.e2e_latency is not None]
        itls = []
        for m in self.metrics:
            itls.extend(m.inter_token_latencies)
        
        total_time = max(m.end_time for m in self.metrics if m.end_time) - \
                     min(m.start_time for m in self.metrics)
        total_output_tokens = sum(m.output_tokens for m in self.metrics)
        
        def percentile(data, p):
            if not data:
                return 0
            sorted_data = sorted(data)
            idx = int(len(sorted_data) * p / 100)
            return sorted_data[min(idx, len(sorted_data) - 1)]
        
        return {
            "num_requests": len(self.metrics),
            "throughput_tok_s": total_output_tokens / total_time if total_time > 0 else 0,
            "throughput_req_s": len(self.metrics) / total_time if total_time > 0 else 0,
            "ttft_avg_ms": statistics.mean(ttfts) * 1000 if ttfts else 0,
            "ttft_p50_ms": percentile(ttfts, 50) * 1000 if ttfts else 0,
            "ttft_p99_ms": percentile(ttfts, 99) * 1000 if ttfts else 0,
            "e2e_avg_ms": statistics.mean(e2e_lats) * 1000 if e2e_lats else 0,
            "e2e_p50_ms": percentile(e2e_lats, 50) * 1000 if e2e_lats else 0,
            "e2e_p99_ms": percentile(e2e_lats, 99) * 1000 if e2e_lats else 0,
            "itl_avg_ms": statistics.mean(itls) * 1000 if itls else 0,
            "itl_p50_ms": percentile(itls, 50) * 1000 if itls else 0,
            "itl_p99_ms": percentile(itls, 99) * 1000 if itls else 0,
        }
    
    def print_report(self):
        s = self.summary()
        print("Performance Report")
        print("=" * 50)
        print(f"  Requests:        {s.get('num_requests', 0)}")
        print(f"  Throughput:      {s.get('throughput_tok_s', 0):.1f} tok/s")
        print(f"                   {s.get('throughput_req_s', 0):.2f} req/s")
        print(f"\n  TTFT (ms):       avg={s.get('ttft_avg_ms', 0):.1f}, "
              f"p50={s.get('ttft_p50_ms', 0):.1f}, p99={s.get('ttft_p99_ms', 0):.1f}")
        print(f"  E2E Latency (ms): avg={s.get('e2e_avg_ms', 0):.1f}, "
              f"p50={s.get('e2e_p50_ms', 0):.1f}, p99={s.get('e2e_p99_ms', 0):.1f}")
        print(f"  ITL (ms):        avg={s.get('itl_avg_ms', 0):.1f}, "
              f"p50={s.get('itl_p50_ms', 0):.1f}, p99={s.get('itl_p99_ms', 0):.1f}")

# Simulate metrics collection
import random
random.seed(42)

collector = ProfilingCollector()
base_time = time.time()

for i in range(20):
    start = base_time + i * 0.5 + random.uniform(0, 0.2)
    prompt_tokens = random.randint(20, 200)
    output_tokens = random.randint(10, 100)
    
    # Simulate token-by-token generation
    ttft = random.uniform(0.05, 0.3)  # 50-300ms TTFT
    first_token = start + ttft
    
    timestamps = [first_token]
    for t in range(output_tokens - 1):
        itl = random.uniform(0.01, 0.05)  # 10-50ms per token
        timestamps.append(timestamps[-1] + itl)
    
    m = RequestMetrics(
        request_id=f"req-{i}",
        prompt_tokens=prompt_tokens,
        output_tokens=output_tokens,
        start_time=start,
        first_token_time=first_token,
        end_time=timestamps[-1],
        token_timestamps=timestamps
    )
    collector.add_metrics(m)

collector.print_report()

## 2. torch.profiler Trace Analysis

In [ ]:
# Demo: Detailed torch.profiler trace analysis workflow

trace_analysis_code = '''
import torch
from torch.profiler import profile, ProfilerActivity, schedule

# Step 1: Capture a trace
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    with_flops=True,  # Estimate FLOPs
) as prof:
    # Run vLLM inference
    output = llm.generate(["Hello world"], SamplingParams(max_tokens=50))

# Step 2: Analyze CPU operations
print("\n=== CPU Time Analysis ===")
print(prof.key_averages().table(
    sort_by="cpu_time_total", row_limit=15
))

# Step 3: Analyze CUDA operations
print("\n=== CUDA Time Analysis ===")
print(prof.key_averages().table(
    sort_by="cuda_time_total", row_limit=15
))

# Step 4: Analyze memory usage
print("\n=== Memory Analysis ===")
print(prof.key_averages().table(
    sort_by="self_cuda_memory_usage", row_limit=15
))

# Step 5: Group by input shapes
print("\n=== By Input Shape ===")
print(prof.key_averages(group_by_input_shape=True).table(
    sort_by="cuda_time_total", row_limit=15
))

# Step 6: Export for visualization
prof.export_chrome_trace("vllm_trace.json")
# Open with: chrome://tracing or https://ui.perfetto.dev

# Step 7: FLOPs analysis
events = prof.key_averages()
for evt in events:
    if evt.flops and evt.flops > 0:
        tflops = evt.flops / evt.cuda_time_total / 1e6 if evt.cuda_time_total > 0 else 0
        print(f"{evt.key}: {tflops:.1f} TFLOPS")
'''

print("torch.profiler Trace Analysis Workflow")
print("=" * 60)
print(trace_analysis_code)

In [ ]:
# Demo: Simulated trace analysis
import random
import math

class TraceEvent:
    def __init__(self, name, cpu_time_us, cuda_time_us, calls, 
                 memory_bytes=0, flops=0):
        self.name = name
        self.cpu_time_us = cpu_time_us
        self.cuda_time_us = cuda_time_us
        self.calls = calls
        self.memory_bytes = memory_bytes
        self.flops = flops

class TraceAnalyzer:
    """Analyzes profiling traces to identify bottlenecks."""
    
    def __init__(self):
        # Simulated trace events for a 7B model inference
        self.events = [
            TraceEvent("aten::mm", 500, 8000, 64, flops=64 * 2 * 4096 * 4096 * 128),
            TraceEvent("aten::scaled_dot_product_attention", 200, 5000, 32, flops=32 * 2 * 128 * 128 * 128),
            TraceEvent("aten::layer_norm", 100, 800, 64),
            TraceEvent("aten::silu", 50, 300, 32),
            TraceEvent("aten::mul", 40, 250, 32),
            TraceEvent("aten::embedding", 30, 100, 1),
            TraceEvent("aten::softmax", 20, 150, 1),
            TraceEvent("aten::copy_", 200, 50, 300, memory_bytes=300 * 1024 * 1024),
            TraceEvent("custom::paged_attention_v2", 100, 4500, 32),
            TraceEvent("custom::rotary_embedding", 50, 400, 32),
            TraceEvent("cudaLaunchKernel", 1000, 0, 500),
            TraceEvent("cudaDeviceSynchronize", 2000, 0, 10),
            TraceEvent("cudaMemcpyAsync", 300, 200, 50),
        ]
    
    def compute_breakdown(self):
        """Compute a breakdown of where time is spent."""
        total_cuda = sum(e.cuda_time_us * e.calls for e in self.events if e.cuda_time_us > 0)
        total_cpu = sum(e.cpu_time_us * e.calls for e in self.events)
        
        categories = {
            "Linear/MatMul": ["aten::mm"],
            "Attention": ["aten::scaled_dot_product_attention", "custom::paged_attention_v2"],
            "Normalization": ["aten::layer_norm"],
            "Activation": ["aten::silu", "aten::mul"],
            "Embedding": ["aten::embedding", "custom::rotary_embedding"],
            "Data Movement": ["aten::copy_", "cudaMemcpyAsync"],
            "Kernel Launch": ["cudaLaunchKernel"],
            "Synchronization": ["cudaDeviceSynchronize"],
            "Other": ["aten::softmax"],
        }
        
        print("Time Breakdown (CUDA)")
        print("=" * 60)
        event_map = {e.name: e for e in self.events}
        
        for cat, ops in categories.items():
            cat_time = sum(
                event_map[op].cuda_time_us * event_map[op].calls
                for op in ops if op in event_map and event_map[op].cuda_time_us > 0
            )
            pct = 100 * cat_time / total_cuda if total_cuda > 0 else 0
            bar = '#' * int(pct / 2)
            print(f"  {cat:20s}: {pct:5.1f}% ({cat_time:10,} us) {bar}")
        
        print(f"\n  Total CUDA time: {total_cuda:,} us")
        print(f"  Total CPU time:  {total_cpu:,} us")
    
    def identify_bottleneck(self):
        """Identify if the workload is compute-bound, memory-bound, or overhead-bound."""
        compute_ops = ["aten::mm", "aten::scaled_dot_product_attention", 
                       "custom::paged_attention_v2"]
        memory_ops = ["aten::copy_", "cudaMemcpyAsync"]
        overhead_ops = ["cudaLaunchKernel", "cudaDeviceSynchronize"]
        
        event_map = {e.name: e for e in self.events}
        
        compute_time = sum(
            event_map[op].cuda_time_us * event_map[op].calls
            for op in compute_ops if op in event_map
        )
        memory_time = sum(
            event_map[op].cuda_time_us * event_map[op].calls
            for op in memory_ops if op in event_map
        )
        overhead_time = sum(
            event_map[op].cpu_time_us * event_map[op].calls
            for op in overhead_ops if op in event_map
        )
        
        total = compute_time + memory_time + overhead_time
        
        print("\nBottleneck Analysis")
        print("=" * 50)
        print(f"  Compute:  {100*compute_time/total:.1f}%")
        print(f"  Memory:   {100*memory_time/total:.1f}%")
        print(f"  Overhead: {100*overhead_time/total:.1f}%")
        
        if compute_time > memory_time and compute_time > overhead_time:
            print("\n  Diagnosis: COMPUTE BOUND")
            print("  Recommendations:")
            print("    - Use faster attention kernels (FlashAttention)")
            print("    - Enable quantization (INT8, FP8)")
            print("    - Use tensor parallelism for larger batch sizes")
        elif memory_time > compute_time:
            print("\n  Diagnosis: MEMORY BOUND")
            print("  Recommendations:")
            print("    - Increase batch size to improve compute/memory ratio")
            print("    - Use quantization to reduce memory transfers")
            print("    - Fuse operations to reduce memory round-trips")
        else:
            print("\n  Diagnosis: OVERHEAD BOUND")
            print("  Recommendations:")
            print("    - Enable CUDA graphs to reduce launch overhead")
            print("    - Reduce synchronization points")
            print("    - Fuse small kernels")

analyzer = TraceAnalyzer()
analyzer.compute_breakdown()
analyzer.identify_bottleneck()

## 3. Nsight Systems Workflow

In [ ]:
# Demo: Nsight Systems profiling workflow

nsight_systems_guide = '''
=======================================================
Nsight Systems (nsys) - System-Level GPU Profiling
=======================================================

Install:
  # Usually comes with CUDA toolkit
  # Or download from: https://developer.nvidia.com/nsight-systems

Basic Profiling:
  nsys profile \\
    --trace=cuda,nvtx,osrt \\
    --output=vllm_profile \\
    --force-overwrite true \\
    python -m vllm.entrypoints.openai.api_server \\
    --model facebook/opt-125m --port 8000

Profile with API markers (NVTX):
  nsys profile \\
    --trace=cuda,nvtx \\
    --nvtx-capture=vllm \\
    --output=vllm_detailed \\
    python script_with_nvtx.py

Delayed start (skip warmup):
  nsys profile \\
    --delay=10 \\
    --duration=30 \\
    --output=vllm_steady_state \\
    python server.py

Key Analysis in Nsight Systems GUI:
  1. Timeline view:
     - CPU threads and their activities
     - GPU kernel execution timeline
     - Memory copies (H2D, D2H, D2D)
     - GPU idle gaps
  
  2. Look for:
     - GPU idle periods (CPU bottleneck)
     - Serial kernel execution (poor overlap)
     - Large memory transfers
     - Synchronization stalls
  
  3. Statistics view:
     - Kernel execution times
     - Memory transfer sizes/durations
     - API call overhead

Command-line statistics:
  nsys stats vllm_profile.nsys-rep --report gputrace
  nsys stats vllm_profile.nsys-rep --report cudaapisum
  nsys stats vllm_profile.nsys-rep --report gpukernsum
'''

print(nsight_systems_guide)

In [ ]:
# Demo: Adding NVTX markers to vLLM/SGLang code

nvtx_code = '''
import torch

# Method 1: Using torch.cuda.nvtx
def model_forward_with_nvtx(model, input_ids, positions, kv_cache):
    """Model forward pass with NVTX annotations."""
    
    torch.cuda.nvtx.range_push("model_forward")
    
    torch.cuda.nvtx.range_push("embedding")
    hidden = model.embed_tokens(input_ids)
    torch.cuda.nvtx.range_pop()
    
    for i, layer in enumerate(model.layers):
        torch.cuda.nvtx.range_push(f"layer_{i}")
        
        torch.cuda.nvtx.range_push("attention")
        hidden = layer.attention(hidden, positions, kv_cache[i])
        torch.cuda.nvtx.range_pop()
        
        torch.cuda.nvtx.range_push("ffn")
        hidden = layer.ffn(hidden)
        torch.cuda.nvtx.range_pop()
        
        torch.cuda.nvtx.range_pop()  # layer_i
    
    torch.cuda.nvtx.range_push("lm_head")
    logits = model.lm_head(hidden)
    torch.cuda.nvtx.range_pop()
    
    torch.cuda.nvtx.range_pop()  # model_forward
    return logits


# Method 2: Using a context manager
from contextlib import contextmanager

@contextmanager
def nvtx_range(name, color=None):
    """NVTX range context manager."""
    torch.cuda.nvtx.range_push(name)
    try:
        yield
    finally:
        torch.cuda.nvtx.range_pop()

# Usage:
def schedule_and_execute():
    with nvtx_range("scheduling"):
        batch = scheduler.schedule()
    
    with nvtx_range("model_forward"):
        output = model.forward(batch)
    
    with nvtx_range("sampling"):
        tokens = sampler.sample(output.logits)
    
    with nvtx_range("detokenize"):
        text = tokenizer.decode(tokens)


# Method 3: Decorator
def nvtx_annotate(name=None):
    def decorator(func):
        label = name or func.__name__
        def wrapper(*args, **kwargs):
            torch.cuda.nvtx.range_push(label)
            try:
                return func(*args, **kwargs)
            finally:
                torch.cuda.nvtx.range_pop()
        return wrapper
    return decorator

@nvtx_annotate("paged_attention")
def paged_attention(query, key_cache, value_cache, ...):
    ...
'''

print("NVTX Annotations for Nsight Systems")
print("=" * 60)
print(nvtx_code)

## 4. Nsight Compute for Kernel Analysis

In [ ]:
# Demo: Nsight Compute workflow for kernel-level analysis

nsight_compute_guide = '''
=======================================================
Nsight Compute (ncu) - CUDA Kernel Profiling
=======================================================

Basic kernel profiling:
  ncu --set full \\
    --target-processes all \\
    --output vllm_kernels \\
    python script.py

Profile specific kernels only:
  ncu --kernel-name "paged_attention" \\
    --launch-count 5 \\
    --set full \\
    python script.py

Profile with metrics:
  ncu --metrics \\
    sm__throughput.avg.pct_of_peak_sustained_elapsed,\\
    dram__throughput.avg.pct_of_peak_sustained_elapsed,\\
    gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed \\
    python script.py

Key Metrics to Examine:
  1. Compute Utilization (SM Throughput)
     - < 60%: Likely memory-bound or launch overhead
     - > 80%: Compute-bound, look at arithmetic intensity
  
  2. Memory Throughput
     - Compare with peak bandwidth
     - High = memory-bound kernel
  
  3. Occupancy
     - Low occupancy may indicate register pressure
     - Or shared memory limitations
  
  4. Warp Stall Reasons
     - Memory dependency: waiting for memory loads
     - Execution dependency: waiting for compute
     - Synchronization: waiting at barrier

Roofline Analysis:
  ncu --set roofline \\
    --kernel-name "attention_kernel" \\
    python script.py
  # This shows where your kernel sits on the roofline model
  # (compute intensity vs achieved performance)
'''

print(nsight_compute_guide)

In [ ]:
# Demo: Simulated kernel analysis and roofline model

class KernelAnalysis:
    """Simulated CUDA kernel analysis for learning."""
    
    def __init__(self, gpu_name="A100"):
        # A100 specs
        self.peak_tflops_fp16 = 312  # TFLOPS
        self.peak_bandwidth_gb = 2039  # GB/s
        self.gpu_name = gpu_name
    
    def analyze_kernel(self, name, flops, bytes_accessed, time_us):
        """Analyze a single kernel."""
        time_s = time_us / 1e6
        achieved_tflops = flops / time_s / 1e12 if time_s > 0 else 0
        achieved_bw = bytes_accessed / time_s / 1e9 if time_s > 0 else 0
        
        arithmetic_intensity = flops / bytes_accessed if bytes_accessed > 0 else 0
        
        # Ridge point: where compute = memory
        ridge_point = self.peak_tflops_fp16 * 1e12 / (self.peak_bandwidth_gb * 1e9)
        
        is_compute_bound = arithmetic_intensity > ridge_point
        
        compute_util = 100 * achieved_tflops / self.peak_tflops_fp16
        bw_util = 100 * achieved_bw / self.peak_bandwidth_gb
        
        print(f"\n  Kernel: {name}")
        print(f"  {'='*50}")
        print(f"  FLOPs:                {flops:,.0f}")
        print(f"  Bytes Accessed:       {bytes_accessed:,.0f}")
        print(f"  Time:                 {time_us:.1f} us")
        print(f"  Achieved TFLOPS:      {achieved_tflops:.1f}")
        print(f"  Achieved Bandwidth:   {achieved_bw:.1f} GB/s")
        print(f"  Arithmetic Intensity: {arithmetic_intensity:.1f} FLOP/byte")
        print(f"  Ridge Point:          {ridge_point:.1f} FLOP/byte")
        print(f"  Compute Utilization:  {compute_util:.1f}%")
        print(f"  Bandwidth Utilization: {bw_util:.1f}%")
        print(f"  Classification:       {'COMPUTE-BOUND' if is_compute_bound else 'MEMORY-BOUND'}")
        
        if is_compute_bound:
            print(f"  Recommendation:       Optimize compute (fuse ops, use tensor cores)")
        else:
            print(f"  Recommendation:       Optimize memory access (coalescing, caching)")

ka = KernelAnalysis()

print(f"GPU: {ka.gpu_name}")
print(f"Peak FP16: {ka.peak_tflops_fp16} TFLOPS")
print(f"Peak BW: {ka.peak_bandwidth_gb} GB/s")

# Analyze typical LLM kernels
kernels = [
    ("GEMM (batch=1, 4096x4096)", 2 * 4096 * 4096 * 1, 4096 * 4096 * 4, 15),
    ("GEMM (batch=32, 4096x4096)", 2 * 4096 * 4096 * 32, 4096 * 4096 * 4, 50),
    ("Paged Attention (bs=1, seqlen=2048)", 2 * 2048 * 128 * 32, 2048 * 128 * 32 * 2, 20),
    ("LayerNorm (4096)", 4096 * 5, 4096 * 4 * 3, 2),
    ("RoPE (4096)", 4096 * 4, 4096 * 4 * 2, 1.5),
]

for name, flops, bytes_acc, time_us in kernels:
    ka.analyze_kernel(name, flops, bytes_acc, time_us)

## 5. Custom Profiling Hooks

In [ ]:
# Demo: Custom profiling hooks for vLLM/SGLang

import time
from contextlib import contextmanager
from collections import defaultdict

class CustomProfiler:
    """Lightweight custom profiler for vLLM/SGLang components."""
    
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.timings = defaultdict(list)
        self.counters = defaultdict(int)
        self._stack = []
    
    @contextmanager
    def region(self, name):
        """Context manager for timing a code region."""
        if not self.enabled:
            yield
            return
        
        full_name = "/".join(self._stack + [name])
        self._stack.append(name)
        start = time.perf_counter()
        try:
            yield
        finally:
            elapsed = time.perf_counter() - start
            self.timings[full_name].append(elapsed)
            self._stack.pop()
    
    def count(self, name, value=1):
        """Increment a counter."""
        self.counters[name] += value
    
    def report(self, top_n=20):
        """Print a profiling report."""
        print("\nCustom Profiling Report")
        print("=" * 80)
        
        # Sort by total time
        sorted_regions = sorted(
            self.timings.items(),
            key=lambda x: sum(x[1]),
            reverse=True
        )[:top_n]
        
        print(f"{'Region':45s} {'Calls':>6s} {'Total(ms)':>10s} "
              f"{'Avg(ms)':>9s} {'Min(ms)':>9s} {'Max(ms)':>9s}")
        print("-" * 90)
        
        for name, times in sorted_regions:
            total = sum(times) * 1000
            avg = total / len(times)
            min_t = min(times) * 1000
            max_t = max(times) * 1000
            print(f"{name:45s} {len(times):6d} {total:10.2f} "
                  f"{avg:9.2f} {min_t:9.2f} {max_t:9.2f}")
        
        if self.counters:
            print("\nCounters:")
            for name, value in sorted(self.counters.items()):
                print(f"  {name}: {value}")
    
    def reset(self):
        self.timings.clear()
        self.counters.clear()

# Simulate profiling a serving loop
profiler = CustomProfiler()

import random
random.seed(42)

for step in range(50):
    with profiler.region("step"):
        # Scheduling
        with profiler.region("scheduling"):
            time.sleep(random.uniform(0.001, 0.005))
            profiler.count("requests_scheduled", random.randint(1, 8))
        
        # Model forward
        with profiler.region("model_forward"):
            # Attention
            with profiler.region("attention"):
                time.sleep(random.uniform(0.005, 0.015))
            # MLP
            with profiler.region("mlp"):
                time.sleep(random.uniform(0.003, 0.010))
        
        # Sampling
        with profiler.region("sampling"):
            time.sleep(random.uniform(0.001, 0.003))
            profiler.count("tokens_generated", random.randint(1, 8))
        
        # Detokenization
        with profiler.region("detokenize"):
            time.sleep(random.uniform(0.0005, 0.002))

profiler.report()

## 6. Case Study: Profiling a Real Workload

In [ ]:
# Demo: Case study - profiling and optimizing a serving workload

class WorkloadSimulator:
    """Simulates profiling a real serving workload with optimization."""
    
    def __init__(self):
        self.configs = {
            "baseline": {
                "attention_ms": 15.0,
                "mlp_ms": 10.0,
                "schedule_ms": 3.0,
                "sampling_ms": 2.0,
                "overhead_ms": 5.0,
                "batch_size": 1,
            },
            "flash_attention": {
                "attention_ms": 5.0,   # 3x faster with FlashAttention
                "mlp_ms": 10.0,
                "schedule_ms": 3.0,
                "sampling_ms": 2.0,
                "overhead_ms": 5.0,
                "batch_size": 1,
            },
            "batched": {
                "attention_ms": 8.0,
                "mlp_ms": 12.0,
                "schedule_ms": 4.0,
                "sampling_ms": 3.0,
                "overhead_ms": 5.0,
                "batch_size": 8,
            },
            "cuda_graphs": {
                "attention_ms": 5.0,
                "mlp_ms": 10.0,
                "schedule_ms": 3.0,
                "sampling_ms": 2.0,
                "overhead_ms": 0.5,  # 10x less overhead
                "batch_size": 1,
            },
            "fully_optimized": {
                "attention_ms": 5.0,
                "mlp_ms": 8.0,    # INT8 quantization
                "schedule_ms": 2.0,
                "sampling_ms": 1.5,
                "overhead_ms": 0.5,
                "batch_size": 8,
            },
        }
    
    def simulate(self, config_name):
        cfg = self.configs[config_name]
        total_per_token = (
            cfg["attention_ms"] + cfg["mlp_ms"] + 
            cfg["schedule_ms"] + cfg["sampling_ms"] + cfg["overhead_ms"]
        )
        tokens_per_second = cfg["batch_size"] * 1000.0 / total_per_token
        
        return {
            "config": config_name,
            "total_per_step_ms": total_per_token,
            "tokens_per_second": tokens_per_second,
            "breakdown": cfg,
        }
    
    def compare_all(self):
        print("Workload Optimization Case Study")
        print("=" * 80)
        print(f"{'Config':25s} {'Step(ms)':>9s} {'Tok/s':>8s} {'Speedup':>8s}  Breakdown")
        print("-" * 80)
        
        baseline_tps = None
        for name in self.configs:
            result = self.simulate(name)
            tps = result["tokens_per_second"]
            if baseline_tps is None:
                baseline_tps = tps
            speedup = tps / baseline_tps
            
            cfg = result["breakdown"]
            breakdown = (f"attn={cfg['attention_ms']:.0f}, mlp={cfg['mlp_ms']:.0f}, "
                        f"sched={cfg['schedule_ms']:.0f}, oh={cfg['overhead_ms']:.1f}, "
                        f"bs={cfg['batch_size']}")
            
            print(f"{name:25s} {result['total_per_step_ms']:9.1f} {tps:8.1f} "
                  f"{speedup:7.1f}x  {breakdown}")

sim = WorkloadSimulator()
sim.compare_all()

---
## Hands-On Assignments

### Assignment 1: Profile a vLLM Serving Workload

**Objective:** Profile the simulated serving workload and generate a report.

In [ ]:
# Assignment 1: Starter Code
import time
import random
import statistics
from collections import defaultdict

class ServingWorkload:
    """Simulated serving workload to profile."""
    
    def __init__(self, model_layers=32, hidden_size=4096):
        self.model_layers = model_layers
        self.hidden_size = hidden_size
        random.seed(42)
    
    def tokenize(self, text):
        time.sleep(random.uniform(0.0005, 0.002))
        return list(range(len(text.split())))
    
    def schedule(self, requests):
        time.sleep(random.uniform(0.001, 0.005))
        return requests[:8]  # Max batch size 8
    
    def model_forward(self, batch_size, seq_len):
        for layer in range(self.model_layers):
            # Attention
            time.sleep(random.uniform(0.0003, 0.0008) * (seq_len / 128))
            # MLP
            time.sleep(random.uniform(0.0002, 0.0005))
    
    def sample(self, batch_size):
        time.sleep(random.uniform(0.0005, 0.002))
        return [random.randint(0, 50000) for _ in range(batch_size)]
    
    def detokenize(self, tokens):
        time.sleep(random.uniform(0.0002, 0.001))
        return ["word" for _ in tokens]


class WorkloadProfiler:
    """TODO: Profile the ServingWorkload and generate a report.
    
    Tasks:
    1. Wrap each method of ServingWorkload with timing code
    2. Run 100 steps of the serving loop
    3. Compute per-component timing breakdown
    4. Identify the top bottleneck
    5. Generate a formatted report
    """
    
    def __init__(self, workload: ServingWorkload):
        self.workload = workload
        self.timings = defaultdict(list)
    
    def profile_step(self, requests):
        """TODO: Profile one serving step."""
        # Time each component:
        # 1. tokenize
        # 2. schedule
        # 3. model_forward
        # 4. sample
        # 5. detokenize
        pass
    
    def run_profile(self, num_steps=100):
        """TODO: Run num_steps of profiling."""
        pass
    
    def generate_report(self):
        """TODO: Generate a formatted profiling report."""
        pass

workload = ServingWorkload()
# profiler = WorkloadProfiler(workload)
# profiler.run_profile(100)
# profiler.generate_report()
print("TODO: Complete the WorkloadProfiler and generate the report.")

### Assignment 2: Identify the Top-3 Bottlenecks

**Objective:** Analyze the provided profiling data and identify the top 3 bottlenecks.

In [ ]:
# Assignment 2: Starter Code

# Simulated profiling data from a real vLLM deployment
profiling_data = {
    "model": "Llama-2-13B",
    "gpu": "A100-80GB",
    "batch_size": 16,
    "avg_prompt_len": 256,
    "avg_output_len": 128,
    "operations": [
        {"name": "linear_projection", "cuda_ms": 45.2, "calls": 160, "type": "compute"},
        {"name": "paged_attention_v2", "cuda_ms": 32.8, "calls": 40, "type": "compute"},
        {"name": "kv_cache_copy", "cuda_ms": 18.5, "calls": 40, "type": "memory"},
        {"name": "layer_norm", "cuda_ms": 8.3, "calls": 80, "type": "compute"},
        {"name": "silu_activation", "cuda_ms": 4.1, "calls": 40, "type": "compute"},
        {"name": "rotary_embedding", "cuda_ms": 3.2, "calls": 40, "type": "compute"},
        {"name": "token_embedding", "cuda_ms": 1.5, "calls": 1, "type": "memory"},
        {"name": "lm_head", "cuda_ms": 5.8, "calls": 1, "type": "compute"},
        {"name": "sampling", "cuda_ms": 2.1, "calls": 1, "type": "compute"},
        {"name": "scheduling", "cpu_ms": 3.5, "calls": 1, "type": "cpu"},
        {"name": "detokenization", "cpu_ms": 1.8, "calls": 16, "type": "cpu"},
        {"name": "kernel_launch_overhead", "cpu_ms": 8.2, "calls": 500, "type": "overhead"},
        {"name": "cuda_sync", "cpu_ms": 5.0, "calls": 5, "type": "overhead"},
    ]
}

class BottleneckAnalyzer:
    """TODO: Analyze profiling data to find bottlenecks.
    
    Tasks:
    1. Compute total time for each operation
    2. Categorize operations (compute, memory, overhead, cpu)
    3. Identify top-3 bottlenecks
    4. For each bottleneck, suggest specific optimizations
    5. Estimate the potential speedup from each optimization
    """
    
    def __init__(self, data):
        self.data = data
    
    def analyze(self):
        """TODO: Perform bottleneck analysis."""
        pass
    
    def suggest_optimizations(self):
        """TODO: Suggest optimizations for each bottleneck."""
        pass
    
    def estimate_speedup(self):
        """TODO: Estimate potential speedup."""
        pass

# analyzer = BottleneckAnalyzer(profiling_data)
# analyzer.analyze()
# analyzer.suggest_optimizations()
# analyzer.estimate_speedup()
print("TODO: Complete the BottleneckAnalyzer class.")
print(f"Data includes {len(profiling_data['operations'])} operations to analyze.")

### Assignment 3: Write a Profiling Report with Recommendations

**Objective:** Generate a comprehensive profiling report.

In [ ]:
# Assignment 3: Starter Code

class ProfilingReport:
    """TODO: Generate a comprehensive profiling report.
    
    The report should include:
    1. Executive Summary (1-2 sentences)
    2. Workload Description (model, hardware, batch size)
    3. Performance Metrics (throughput, latency)
    4. Bottleneck Analysis (with percentages)
    5. Top-3 Recommendations (with expected impact)
    6. Detailed Breakdown Table
    """
    
    def __init__(self, profiling_data, metrics):
        self.data = profiling_data
        self.metrics = metrics
    
    def generate(self):
        """TODO: Generate the complete report."""
        sections = [
            self._executive_summary(),
            self._workload_description(),
            self._performance_metrics(),
            self._bottleneck_analysis(),
            self._recommendations(),
            self._detailed_breakdown(),
        ]
        return "\n\n".join(s for s in sections if s)
    
    def _executive_summary(self):
        """TODO: Write executive summary."""
        return None
    
    def _workload_description(self):
        """TODO: Describe the workload."""
        return None
    
    def _performance_metrics(self):
        """TODO: Report key metrics."""
        return None
    
    def _bottleneck_analysis(self):
        """TODO: Analyze bottlenecks."""
        return None
    
    def _recommendations(self):
        """TODO: Provide recommendations."""
        return None
    
    def _detailed_breakdown(self):
        """TODO: Detailed timing breakdown."""
        return None

# Sample metrics to use in the report
sample_metrics = {
    "throughput_tok_s": 450.0,
    "ttft_avg_ms": 120.0,
    "ttft_p99_ms": 350.0,
    "itl_avg_ms": 25.0,
    "itl_p99_ms": 45.0,
    "gpu_util_pct": 72.0,
    "memory_used_gb": 52.0,
    "memory_total_gb": 80.0,
}

# report = ProfilingReport(profiling_data, sample_metrics)
# print(report.generate())
print("TODO: Complete the ProfilingReport class.")
print("Generate a comprehensive, actionable profiling report.")

---
## Summary

In this notebook, we covered:

1. **Profiling methodology** - what to measure and when for LLM serving
2. **torch.profiler** - trace analysis with CPU and CUDA timing
3. **Nsight Systems** - system-level GPU profiling with NVTX markers
4. **Nsight Compute** - kernel-level analysis and roofline model
5. **Custom profiling hooks** - lightweight instrumentation for serving loops
6. **Case study** - profiling and optimization workflow end-to-end

**Next:** Chapter 11.5 - Code Review Standards & PR Process